# 16 analysis — LoCoMo full context vs the model window

Frozen: LoCoMo questions + `qa_mem0_v1`. Variable: how much of the **conversation** is injected as `{memory}`.

**Verified meaning of `full_context`:** in the Mem0 paper and in this repo it is the entire conversation attached to the current LoCoMo question. It is **not** the LoCoMo dataset, and it is **not** “fill the model’s maximum context window.” Mem0 Table 2 lists **26,031** `memory_tokens` for that baseline. Our `full_context_qa` clone bills ~27.8k mean prompt tokens on the same 10 conversations.

A context window is the maximum tokens (input + output, and reasoning when the model thinks) one request may use. Providers reject or truncate beyond that. We do not pad to the limit. LoCoMo eval already knows which conversation a question belongs to; retrieval never searches the other nine conversations.

| Layer | What it is |
|-------|------------|
| Dataset | LoCoMo → 10 conversations → sessions → turns; questions sit on one conversation |
| Full context | Raw timestamped transcript of that conversation |
| Sessions | Same conversation, one summary block per session (all sessions, compressed) |
| RAG | Chunk the transcript (256 tokens), embed, retrieve top k∈{1,2} |
| Mem0 / Mem0g | Extract facts (Mem0g adds graph relations); retrieve inside the conversation |

Engine: `python -m scripts.analysis.context_window` (not a memorybench job). Notebook stays a thin wrapper — no matplotlib or groupby here.


In [ ]:
from pathlib import Path
import sys

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.experiment_runner.analysis.context_window import (
    notebook_show_context_window,
    render_context_window_report,
)

report = render_context_window_report(root=ROOT)
print(report.out_dir)
print("skipped:", report.skipped_packs)
print("cells:", len(report.cell_table), "conversations:", len(report.conversation_table))


## How the window is used

`FullContextMemoryBuilder` joins every turn as `{timestamp} | {speaker}: {text}` and fills `prompts/readers/qa_mem0_v1.txt`. One Chat Completions call per question. `max_chars` is null, so nothing is trimmed to the window. Completion caps (64 / 256 / 8192) only bound the **answer**, not the transcript.

Local audit (`experiments/full_context_qa`): 10 conversations, 19–32 sessions, ~17k–33k billed input. That is ~22% of gpt-4o-mini’s 128k window and ~3% of GPT-5.6 Terra’s 1.05M window. 2025→2026 **utilization fell** because windows grew and LoCoMo transcripts did not.

Session methods still pass every session. RAG does not — two 256-token chunks (~0.9k billed). Scores track that coverage gap, not unused window. J per 1k input tokens makes RAG look “efficient”; category holes show it is under-covered. Teacher session summaries keep session coverage at ~5–8k tokens (~20% of FC) with a frozen gpt-4o-mini reader.


In [ ]:
notebook_show_context_window(report)


## Reading utilization, efficiency, and year moves

- **Utilization** = billed prompt tokens / published window. Full-context fill is 14–26% of 128k and 2–3% of 1M. Idle tokens are not extra evidence.
- **Coverage** = input tokens / same-reader full-context mean (paper pin 26,031 if no FC cell). RAG ≈ 0.03; teacher sessions ≈ 0.2; FC = 1.0.
- **Efficiency** = J per 1k input, J per second, J per pinned USD. RAG wins token-efficiency and loses J. Thinking-on raises FC J more than RAG J because the transcript is already in the prompt.
- **Year axis:** same ~28k transcript; 2024 128k → 2025 400k → 2026 1M. Utilization drops; FC vs RAG ranking does not flip because of that drop.
- Literature J for Full-context is a Table 2 pin (72.90), not this OSS clone. Local `full_context_qa` J is 74.68% on gpt-4o-mini with a different stack than the paper’s gpt-4o-class run.

Plots: `experiments/_campaign/context_window/plots/` (window fill, input by method/conversation, coverage vs J/latency/USD, year utilization and J per 1k).
